# UK Constituency Boundaries — Catalogue

Combined build notebook for Westminster, Scottish Parliament (Holyrood), and
Senedd Cymru constituency boundaries, plus Northern Ireland (in progress).

Each section:
1. Pulls boundaries from the relevant ArcGIS FeatureServer
2. Renames columns to something legible, while keeping a documented mapping
   back to the original source field names
3. Saves to its own GeoPackage file, with a `source_metadata` table written
   into the same file for provenance

**Note:** each geography is saved to its own `.gpkg` file (not combined into
one), so metadata is written per-file rather than as a single shared table.


## Setup

Shared imports and the ArcGIS fetch helper.

**Note:** `gpd.read_file(url)` should be avoided for these ArcGIS
FeatureServer query endpoints — it triggers an HTTP range-request probe
that Esri's REST services don't handle well, which surfaces as a
confusing `504 Gateway Timeout` even when the service is actually up.
Fetching the JSON manually with `requests` and handing the parsed
features to geopandas avoids this.


In [1]:
import requests
import geopandas as gpd
import pandas as pd
import sqlite3
import json


def fetch_arcgis_layer(base_url, page_size=2000, timeout=60):
    """base_url should point at .../FeatureServer/<layer_id> (no trailing /query)."""
    offset = 0
    frames = []
    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }
        r = requests.get(f"{base_url}/query", params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        if not data.get("features"):
            break
        frames.append(gpd.GeoDataFrame.from_features(data["features"]))
        if len(data["features"]) < page_size:
            break
        offset += page_size
    return pd.concat(frames, ignore_index=True)


## Westminster Parliamentary Constituencies (July 2024, UK BFC)

**Note:** the live `/FeatureServer/0/query` endpoint for this layer timed out repeatedly (504 Gateway Timeout) confirmed as a service-side issueon the query operation specifically, not a code problem, since even a5-record, no-geometry request timed out from multiple independentfetches. The `/MapServer/0` endpoint on the same host loaded instantly, which showed the host itself was up but only had `"capabilities": "Map"` no query support, so it's not a usable substitute.

**Fix:** downloaded the static GeoJSON export from the ONS Open Geography Portal instead of hitting the live query endpoint. Static exports are served from storage rather than generated per-request, so they don't hit the same timeout issue.

Source page: https://geoportal.statistics.gov.uk/datasets/ons::westminster-parliamentary-constituencies-july-2024-boundaries-uk-bfc-2/about


In [3]:
gdf = gpd.read_file(r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\raw\Westminster_Parliamentary_Constituencies_July_2024_Boundaries_UK_BFC_-1420043245393085943.gpkg")
print(gdf.columns)


Index(['PCON24CD', 'PCON24NM', 'PCON24NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT',
       'GlobalID', 'geometry'],
      dtype='str')


**Note:** original columns were
`['PCON24CD', 'PCON24NM', 'PCON24NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']`
-- this static export doesn't include the `FID`/`Shape__Area`/`Shape__Length`
bookkeeping fields that the live query returns (those are ArcGIS-computed,
not source attributes).

`PCON24NMW` is the Welsh name field -- populated for Welsh constituencies
only, null elsewhere. That's expected, not missing data. This layer covers
the *whole* UK (England, Scotland, Wales, NI all return MPs to Westminster),
not just England.


In [4]:
column_rename = {
    "PCON24CD":  "constituency_code",     # ONS 9-char GSS code, e.g. E14001234
    "PCON24NM":  "constituency_name",
    "PCON24NMW": "constituency_name_welsh",
    "BNG_E":     "easting_bng",           # British National Grid easting (centroid)
    "BNG_N":     "northing_bng",          # British National Grid northing (centroid)
    "LONG":      "longitude",             # WGS84 centroid longitude
    "LAT":       "latitude",              # WGS84 centroid latitude
    "GlobalID":  "global_id",
}

westminster = gdf.rename(columns=column_rename)
westminster.head()


,constituency_code,constituency_name,constituency_name_welsh,easting_bng,northing_bng,longitude,latitude,global_id,geometry
0,E14001063,Aldershot,,484716,155270,-0.78648,51.290272,{32D9B16C-4564-48BC-96A1-088F823FA507},"MULTIPOLYGON (((485406.902 159918.603, 485403...."
1,E14001064,Aldridge-Brownhills,,404720,301030,-1.93172,52.607040,{D80A7CD2-F8C8-4097-9A87-C888F3925E78},"MULTIPOLYGON (((406519.098 305054.298, 406507...."
2,E14001065,Altrincham and Sale West,,374132,389051,-2.39049,53.397659,{45EA1B6A-DFC6-4AB4-86A1-786BD13F47D0},"MULTIPOLYGON (((379104.096 393143.903, 379101...."
3,E14001066,Amber Valley,,440478,349674,-1.39771,53.042820,{00E7C6D5-EB09-40D5-B5A8-71BC3D90B943},"MULTIPOLYGON (((444868.402 353958.1, 444851.69..."
4,E14001067,Arundel and South Downs,,497309,118530,-0.61584,50.957981,{CC0A5F40-A480-4527-979F-BE7455F25DB8},"MULTIPOLYGON (((505813.9 133399.4, 505813.1 13..."


In [5]:
westminster_metadata = {
    "layer": "westminster_pcon_jul2024",
    "source_name": "PCON_JULY_2024_UK_BFC \u2014 Westminster Parliamentary Constituencies",
    "source_url": "https://geoportal.statistics.gov.uk/datasets/ons::westminster-parliamentary-constituencies-july-2024-boundaries-uk-bfc-2/about",
    "publisher": "Office for National Statistics (ONS) Open Geography Portal",
    "boundary_type": "BFC \u2014 Full resolution, clipped to coastline (Mean High Water mark)",
    "as_at_date": "2024-07-04",
    "retrieved_date": "2026-07-20",
    "review_context": "2023 Boundary Commission review, effective at the July 2024 UK general election",
    "crs": str(westminster.crs),
    "n_features": len(westminster),
    "dropped_fields": [],  # static GeoJSON export did not include FID/Shape__Area/Shape__Length
    "original_to_renamed_columns": column_rename,
    "notes": "PCON24NMW (Welsh name) is populated for Welsh constituencies only; null elsewhere -- expected, not missing data. Covers the whole UK (England, Scotland, Wales, NI), not just Westminster-in-England.",
}


In [7]:
westminster.to_file(r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\processed\westminster_2024.gpkg", layer="westminster_pcon_jul2024", driver="GPKG")

## Scottish Parliament Constituencies (Holyrood, May 2026, SC BFC)

**Note:** the Scottish Parliament sits in Edinburgh, entirely separate from Westminster Holyrood constituencies are drawn independently by Boundaries Scotland and don't share boundaries with Westminster constituencies (73 Holyrood constituencies + 8 regions vs. 57 Scottish Westminster seats, historically; boundaries diverge further with each independent review).

This layer (`SPC_MAY_2026_SC_BFC`) reflects the **Second Review of Scottish Parliament Boundaries** -- final recommendations approved 15 Oct 2025 (SSI 2025/285), taking effect at the next Scottish Parliament election, scheduled 7 May 2026.

**Note:** found a second, related service on the same host `SPC_DEC_2025_SC_NC`, a names-and-codes lookup table with fields`SPC25CD`/`SPC25NM` (December 2025 vintage) rather than`SPC26CD`/`SPC26NM` (May 2026 boundary layer). If cross-referencingagainst a names-and-codes table later, check the vintage matches otherwise codes will silently mismatch.

Source: https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/SPC_MAY_2026_SC_BFC/FeatureServer/0


In [9]:
holyrood = fetch_arcgis_layer(
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/"
    "SPC_MAY_2026_SC_BFC/FeatureServer/0"
)
print(holyrood.columns)


KeyboardInterrupt: 

**Note:** original columns were
`['geometry', 'FID', 'SPC26CD', 'SPC26NM', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'Shape__Area', 'Shape__Length', 'GlobalID']`
-- pulled from the live query this time (unlike Westminster), so it
includes the three ArcGIS bookkeeping fields (`FID`, `Shape__Area`,
`Shape__Length`) that the Westminster static export dropped. Removed
below for consistency.

No Gaelic name field -- unlike Westminster's `PCON24NMW` for Welsh,
`SPC26NM` is the only name field. Expected, not a gap.


In [ ]:
holyrood = holyrood.drop(columns=["FID", "Shape__Area", "Shape__Length"])

column_rename = {
    "SPC26CD":  "constituency_code",     # Scottish Parliament boundary set, effective May 2026 (post Second Review)
    "SPC26NM":  "constituency_name",
    "BNG_E":    "easting_bng",           # British National Grid easting (centroid)
    "BNG_N":    "northing_bng",          # British National Grid northing (centroid)
    "LONG":     "longitude",             # WGS84 centroid longitude
    "LAT":      "latitude",              # WGS84 centroid latitude
    "GlobalID": "global_id",
}

holyrood = holyrood.rename(columns=column_rename)
holyrood.head()


In [ ]:
holyrood_metadata = {
    "layer": "scottish_parliament_spc_may2026",
    "source_name": "SPC_MAY_2026_SC_BFC \u2014 Scottish Parliamentary Constituencies",
    "source_url": "https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/SPC_MAY_2026_SC_BFC/FeatureServer/0",
    "publisher": "Boundaries Scotland / ONS-hosted ArcGIS service",
    "boundary_type": "BFC \u2014 Full resolution, clipped to coastline (Mean High Water mark)",
    "as_at_date": "2026-05-07",  # takes effect at the next Scottish Parliament election
    "retrieved_date": "2026-07-20",
    "review_context": "Second Review of Scottish Parliament Boundaries \u2014 final recommendations approved 15 Oct 2025 (SSI 2025/285), effective at the 7 May 2026 election",
    "crs": str(holyrood.crs),
    "n_features": len(holyrood),
    "dropped_fields": ["FID", "Shape__Area", "Shape__Length"],  # ArcGIS-computed bookkeeping, not source attributes
    "original_to_renamed_columns": column_rename,
    "notes": "No Gaelic name field present (unlike Westminster's PCON24NMW for Welsh) -- SPC26NM is the only name field. A separate SPC_DEC_2025_SC_NC names-and-codes table exists on the same host but is a different vintage (SPC25 codes) -- do not cross-reference without checking alignment.",
}


In [ ]:
holyrood.to_file(r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\processed\holyrood_2026.gpkg", layer="holyrood_2026", driver="GPKG")

## Senedd Cymru Constituencies (May 2026, WA BFC)

**Note:** confirmed via search that these are **16 six-member
constituencies**, not the old 40 single-member constituencies, and not
"regions" either (the old 5 electoral regions are abolished, not renamed).

Under the **Senedd Cymru (Members and Elections) Act 2024**:
- 96 total Members (up from 60)
- 16 constituencies, each electing 6 Members via closed-list proportional
  representation (D'Hondt formula)
- Boundaries created by the Democracy and Boundary Commission Cymru,
  formed by pairing the 32 UK Parliamentary (Westminster) constituencies
  in Wales -- Final Determinations published March 2025
- First used at the 2026 Senedd election (7 May 2026)

So `constituency_code`/`constituency_name` is the right terminology after
all (not `region_code`/`region_name` as first guessed when the count of
16 came back) -- but each polygon here represents a **6-member**
constituency, unlike the single-member Westminster/Holyrood constituencies
in this catalogue. Worth flagging clearly wherever this layer is used
downstream, since a naive row-for-row comparison across layers would be
misleading (one Senedd polygon = 6 seats, one Westminster/Holyrood polygon
= 1 seat).

Source: https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/SENC_MAY_2026_WA_BFC/FeatureServer/0


In [10]:
senedd = fetch_arcgis_layer(
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "SENC_MAY_2026_WA_BFC/FeatureServer/0"
)
print(len(senedd))
senedd.head()


16


,geometry,FID,SENC26CD,SENC26NM,BNG_E,BNG_N,LONG,LAT,Shape__Area,Shape__Length,GlobalID
0,"POLYGON ((-3.58221 51.71404, -3.58147 51.71396...",1,W09000048,Afan Ogwr Rhondda,288501,191452,-3.611638,51.61083,4.415283e+08,135619.653844,e670ce91-4d08-417d-a01e-0ac2e9800e86
1,"MULTIPOLYGON (((-4.40784 53.14366, -4.40775 53...",2,W09000049,Bangor Conwy Môn,285758,362572,-3.709626,53.14801,2.113214e+09,688409.301915,83aa25b8-b513-47bf-84fe-eb01763aeab1
2,"POLYGON ((-3.15735 51.81605, -3.15728 51.81593...",3,W09000050,Blaenau Gwent Caerffili Rhymni,316297,190529,-3.210106,51.60735,3.072697e+08,136691.965383,945cdf84-d2c3-432b-a250-65cd59d89c1d
3,"POLYGON ((-3.23087 52.45323, -3.23104 52.45322...",4,W09000051,Brycheiniog Tawe Nedd,300684,246320,-3.451543,52.10627,3.313635e+09,457181.993222,2ad94b36-4e37-4c33-8540-ddf326073a6b
4,"POLYGON ((-3.26429 51.56918, -3.26399 51.56898...",5,W09000052,Caerdydd Ffynnon Taf,319181,181303,-3.166344,51.52484,7.859142e+07,60997.809397,875294eb-d63c-4e41-a121-0d661eb87048


**Note:** original columns were
`['geometry', 'FID', 'SENC26CD', 'SENC26NM', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'Shape__Area', 'Shape__Length', 'GlobalID']`
-- same shape as Holyrood (live query, so includes the ArcGIS bookkeeping
fields). `len(senedd)` returned 16, confirming this is the new
constituency set.


In [ ]:
senedd = senedd.drop(columns=["FID", "Shape__Area", "Shape__Length"])

column_rename = {
    "SENC26CD": "constituency_code",
    "SENC26NM": "constituency_name",
    "BNG_E":    "easting_bng",
    "BNG_N":    "northing_bng",
    "LONG":     "longitude",
    "LAT":      "latitude",
    "GlobalID": "global_id",
}

senedd = senedd.rename(columns=column_rename)
senedd.head()


In [15]:
senedd_metadata = {
    "layer": "senedd_constituencies_may2026",
    "source_name": "SENC_MAY_2026_WA_BFC \u2014 Senedd Constituencies",
    "source_url": "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/SENC_MAY_2026_WA_BFC/FeatureServer/0",
    "publisher": "Democracy and Boundary Commission Cymru / ONS-hosted ArcGIS service",
    "boundary_type": "BFC \u2014 Full resolution, clipped to coastline (Mean High Water mark)",
    "as_at_date": "2026-05-07",
    "retrieved_date": "2026-07-20",
    "review_context": "Senedd Cymru (Members and Elections) Act 2024 reform \u2014 Wales moved from 40 single-member constituencies + 5 regions (60 Members total) to 16 six-member constituencies (96 Members total), closed-list D'Hondt proportional representation, created by pairing the 32 Welsh Westminster constituencies. Boundaries confirmed by the Democracy and Boundary Commission Cymru's Final Determinations, March 2025. First used at the 2026 Senedd election (7 May 2026).",
    "crs": str(senedd.crs),
    "n_features": len(senedd),
    "dropped_fields": ["FID", "Shape__Area", "Shape__Length"],
    "original_to_renamed_columns": column_rename,
    "notes": "16 features confirmed as six-member constituencies (not single-member, not 'regions' -- the old 5 electoral regions were abolished, not renamed). Each polygon here represents 6 Senedd seats, unlike the single-member Westminster/Holyrood layers in this catalogue -- do not treat rows as directly comparable across layers without accounting for this.",
}


In [16]:
senedd.to_file(r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\processed\senedd_2026.gpkg", layer="senedd_2026", driver="GPKG")


c:\Users\spspa\miniforge3\envs\raster_env\Lib\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


## Northern Ireland Assembly Constituencies -- IN PROGRESS

**Note:** The NI Assembly currently uses the same 18 constituencies
as Westminster (2023 Boundary Commission for Northern Ireland review,
effective from the 2024 Westminster election).

**Historical caveat (worth keeping in the write-up):** this alignment is
*current*, not structurally guaranteed NI Assembly and Westminster
boundaries have diverged before when the two boundary-review processes
ran on different schedules (e.g. Belfast West's boundaries differed from
its Westminster equivalent in 1973-74, 1983-86, and 2010-11; North Antrim
wasn't updated in the 1996-97 redistribution even as Westminster's
boundaries changed). So "identical" should be read as "identical as of
this dataset's vintage," not a permanent fact.
